In [ ]:
# STRUCTURE:
# actual results, and initial stop (if never moved)
# no stop
# wide stop / max tolerable loss (20-40% range)
# 8, 10, 12.5, 15% stops
# 2, 2.5, 3 ATR stops
# maybe daily higher low and weekly higher low pivot stops, if I can figure that out, in a more advanced version

# ASSUMPTIONS:
# Entry is taken around the close of the day, so the first day is skipped when assessing R-multiples. Hence df.iloc[1:] syntax.
# On days where price gaps down below my stop, the stop is triggered at the open. Not always going to be the case
# No commissions or slippage on exit, so a round -1R loss if stop is hit intraday. Would like to fix in a future version.
# Intraday action is currently ignored. This may be an issue on days where my stop hits before a new high for the move. Will probably address with intraday price data and resampling. 

# EDGE CASES:
# Think about intraday entry and exit timing. E.g. what if my stop is below the low of the day, but I enter after the low of the day is set? Does it matter if entry is at the close?
# what if there's no data from df.iloc[1:]? 
# what should realised_r return if not stopped out? na value?
# what happens on a day if my stop hits before the high of the day? 

# FUTURE CONSIDERATIONS:
# Would I go about this process a different way with a larger dataset? Does pandas have a built in function, so I don't have to use the slower python loops?
# Instead of looping yfinance for every trade, should I store price data in a csv? Or would that be unnecessary? Depends on speed, data accuracy, etc. 
# Should I add some kind of drawdown calculation in a future version, so I can see what type of pullbacks I might expect and how viable wide stops actually are.
# While this isn't a consideration for v1, for future versions, I'd like to consider intraday stops, slippage, commissions,etc. I want to make this professional-grade.
# At some point I plan to test trailing stops on profitable trades to see what the most effective method there would be. E.g. pivot lows, moving average, atr, percentage trailing, etc.
# Worth adding MAE and MFE at some stage, to see how much a position goes for and against me before resolution. 
# I think I should replace exit data and exit price with last_date and last_price (or final) and then add a boolean column for stopped_out (True/False), would help with available_r. 
# no_stop_max_r can be derived from this using the trade method's 1r value, if desired...
# Add realised_r (or available_r for open trades) and max_r back for each trade, and think about no_stop_max_r comparisons.
# I think max drawdown is going to be important when comparing stop loss types. E.g. A wider stop might look better on paper, but is the drop tolerable psychologically?

# TASKS:
# Ran first analysis. I think I'd like more context. E.g position sizing and average $ return per stop type. Perhaps also risk-adjusted returns using r-multiples calculations. 
# Create atr_stop functions, and repeat pandas analysis. 
# Try figuring out the daily and weekly pivot stops. If it's not getting anywhere, save for future version and move on to Pardo. 

# ISSUES:
# Minor inconsistency in functions and order of ticker/date return.
# Is it an unfair test? My current setup does not consider losses. 

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [11]:
# pandas loop helper function

def trade_loop(data, entry_price, stop_price):

    max_price = entry_price
    exit_date = np.nan
    exit_price = np.nan

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                exit_price = row['Open']
                
            else:
                exit_price = stop_price
            
            exit_date = row_name.date()
            break

    if not pd.isna(exit_price):
        exit_price = round(exit_price, 2)

    max_pct = ((max_price - entry_price) / entry_price) * 100
    max_pct = round(max_pct, 2)

    return {'exit_date':exit_date, 
            'exit_price': exit_price, 
            'max_pct': max_pct}


In [12]:
# baseline_stop helper function, calculates the max tolerable loss based on given baseline_stop_pct

def baseline_stop(data, entry_price, baseline_stop_pct): 

    baseline_stop_price = entry_price * (1 - (baseline_stop_pct / 100))

    baseline_loop = trade_loop(data, entry_price, baseline_stop_price)
    baseline_max_pct = baseline_loop['max_pct']

    return baseline_max_pct

In [13]:
# False negative helper function, checks whether stop prevented trade from capturing sufficient portion of fat tail

def false_negative_test(max_pct, baseline_max_pct, rally_pct_threshold, tail_pct_threshold):

    if  baseline_max_pct < 0.01:
        tail_pct = np.nan
    else: 
        tail_pct = (max_pct / baseline_max_pct) * 100
        tail_pct = round(tail_pct, 2)

    if tail_pct > 100:
         tail_pct = 100
    
    if (baseline_max_pct >= rally_pct_threshold) and (tail_pct < tail_pct_threshold):
            return {'bool':True, 
                    'tail_pct': tail_pct}

    return {'bool':False, 
            'tail_pct': tail_pct}

In [ ]:
# Position sizing helper function

def position_sizing(entry_price, stop_price, max_pct, account_size, portfolio_risk):

    dollar_risk = account_size * portfolio_risk
    trade_risk = 1 - (stop_price / entry_price)

    position_size = dollar_risk / trade_risk

    appreciation_value = position_size * (1 + (max_pct / 100))

    max_trade_ret = appreciation_value - position_size
    
    position_size = round(position_size, 2)
    max_trade_ret = round(max_trade_ret, 2)

    return {'position_size':position_size, 
            'max_trade_ret': max_trade_ret}

In [15]:
# Actual trade result function

def actual_outcome(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):

    max_price = trade_row['entry_price']
    
    date = pd.to_datetime(trade_row['entry_date']) + pd.DateOffset(days=1) # Possible calendar date issue to be aware of here. We want trading days...
    
    if not pd.isna(trade_row['exit_date']):
        for row_name, row in data.loc[date:trade_row['exit_date']].iterrows(): # Ideally would use iloc[1: ...] as elsewhere...

            if row['High'] > max_price:
                max_price = row['High']
    else:
        for row_name, row in data.iloc[1:].iterrows():

            if row['High'] > max_price:
                max_price = row['High']

    max_pct = ((max_price - trade_row['entry_price']) / trade_row['entry_price']) *100
    max_pct = round(max_pct, 2)

    false_negative = false_negative_test(max_pct, baseline_max_pct, rally_pct_threshold, tail_pct_threshold)

    sizing = position_sizing(trade_row['entry_price'], trade_row['stop_price'], max_pct, account_size, portfolio_risk)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': trade_row['stop_price'],
            'exit_date': trade_row['exit_date'],
            'exit_price': trade_row['exit_price'],
            'max_pct': max_pct,
            'baseline_max_pct': baseline_max_pct,
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'position_sizing': sizing['position_size'], 
            'max_trade_ret': sizing['max_trade_ret'],
            'stop_type': 'Actual Outcome'}

In [ ]:
# Initial stop result function

def initial_stop(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):
    
    trade = trade_loop(data, trade_row['entry_price'], trade_row['stop_price'])

    false_negative = false_negative_test(trade['max_pct'], baseline_max_pct, rally_pct_threshold, tail_pct_threshold)

    sizing = position_sizing(trade_row['entry_price'], trade_row['stop_price'], trade['max_pct'], account_size, portfolio_risk)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': trade_row['stop_price'],
            'exit_date': trade['exit_date'],
            'exit_price': trade['exit_price'],
            'max_pct': trade['max_pct'],
            'baseline_max_pct': baseline_max_pct,
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'position_sizing': sizing['position_size'], 
            'max_trade_ret': sizing['max_trade_ret'],
            'stop_type': 'Initial Stop'}

In [39]:
# Percentage stop function

def pct_stop(data, trade_row, stop_pct, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):

    stop_price = trade_row['entry_price'] * (1 - (stop_pct / 100))
    stop_price = round(stop_price, 2)

    trade = trade_loop(data, trade_row['entry_price'], stop_price)

    false_negative = false_negative_test(trade['max_pct'], baseline_max_pct, rally_pct_threshold, tail_pct_threshold)

    sizing = position_sizing(trade_row['entry_price'], stop_price, trade['max_pct'], account_size, portfolio_risk)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': stop_price,
            'exit_date': trade['exit_date'],
            'exit_price': trade['exit_price'],
            'max_pct': trade['max_pct'],
            'baseline_max_pct': baseline_max_pct,
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'position_sizing': sizing['position_size'], 
            'max_trade_ret': sizing['max_trade_ret'],
            'stop_type': f'{stop_pct}_pct Stop'}

In [36]:
# Stop losses simulation function

def stop_sim(trade_row, stop_pct, baseline_stop_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk):
    
    data = yf.download(trade_row['ticker'], start=trade_row['entry_date'], multi_level_index=False, auto_adjust=True, progress=False)

    baseline_max_pct = baseline_stop(data, trade_row['entry_price'], baseline_stop_pct)

    sim_results = []
    sim_results.append(actual_outcome(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))
    sim_results.append(initial_stop(data, trade_row, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))
    
    for i in stop_pct:
        sim_results.append(pct_stop(data, trade_row, i, baseline_max_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))

    return sim_results


In [40]:
# Reading trades CSV file, then using it as input for a list of dicts, which stores trade outputs.

trades = pd.read_csv('trades.csv')
stop_pct = [8, 10, 12.5, 15, 20, 25, 30]
baseline_stop_pct = 40
rally_pct_threshold = 30
tail_pct_threshold = 40
account_size = 100000
portfolio_risk = 0.02

results = []

for row_name, row in trades.iterrows():
    results.extend(stop_sim(row, stop_pct, baseline_stop_pct, rally_pct_threshold, tail_pct_threshold, account_size, portfolio_risk))


In [41]:
# List of nested dicts converted into dataframe

results_df = pd.DataFrame(results)
results_df

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,max_pct,baseline_max_pct,tail_pct,false_negative,position_sizing,max_trade_ret,stop_type
0,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.05,2.39,46.35,5.16,True,32947.32,787.44,Actual Outcome
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,2.39,46.35,5.16,True,32947.32,787.44,Initial Stop
2,2025-08-12,TSLA,340.84,313.57,NaN,NaN,46.35,46.35,100.00,False,24997.43,11586.31,8_pct Stop
3,2025-08-12,TSLA,340.84,306.76,NaN,NaN,46.35,46.35,100.00,False,20002.35,9271.09,10_pct Stop
4,2025-08-12,TSLA,340.84,298.23,NaN,NaN,46.35,46.35,100.00,False,15998.12,7415.13,12.5_pct Stop
...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,2026-04-08,STX,495.76,433.79,NaN,NaN,69.70,69.70,100.00,False,16000.00,11152.00,12.5_pct Stop
293,2026-04-08,STX,495.76,421.40,NaN,NaN,69.70,69.70,100.00,False,13334.05,9293.83,15_pct Stop
294,2026-04-08,STX,495.76,396.61,NaN,NaN,69.70,69.70,100.00,False,10000.20,6970.14,20_pct Stop
295,2026-04-08,STX,495.76,371.82,NaN,NaN,69.70,69.70,100.00,False,8000.00,5576.00,25_pct Stop


In [42]:
results_df.loc[results_df['stop_type'] == '30_pct Stop']

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,max_pct,baseline_max_pct,tail_pct,false_negative,position_sizing,max_trade_ret,stop_type
8,2025-08-12,TSLA,340.84,238.59,NaN,NaN,46.35,46.35,100.00,False,6666.80,3090.06,30_pct Stop
17,2025-08-26,STX,165.36,115.75,NaN,NaN,408.77,408.77,100.00,False,6666.40,27250.24,30_pct Stop
26,2025-08-26,CCL,31.89,22.32,NaN,NaN,5.57,5.57,100.00,False,6664.58,371.22,30_pct Stop
35,2025-08-28,FLEX,54.77,38.34,NaN,NaN,169.02,169.02,100.00,False,6667.07,11268.68,30_pct Stop
44,2025-08-28,BROS,74.22,51.95,2025-09-29,51.95,0.01,0.01,100.00,False,6665.47,0.67,30_pct Stop
53,2025-09-05,MU,131.30,91.91,NaN,NaN,523.51,523.51,100.00,False,6666.67,34900.68,30_pct Stop
62,2025-09-11,NET,225.50,157.85,NaN,NaN,15.30,15.30,100.00,False,6666.67,1020.00,30_pct Stop
71,2025-09-11,RDDT,261.15,182.80,2025-11-06,182.80,8.35,8.35,100.00,False,6666.24,556.63,30_pct Stop
80,2025-09-11,TSLA,367.85,257.50,NaN,NaN,35.61,35.61,100.00,False,6666.97,2374.11,30_pct Stop
89,2025-09-15,CCJ,86.23,60.36,NaN,NaN,56.84,56.84,100.00,False,6666.41,3789.19,30_pct Stop


In [51]:
# Stop loss analysis using pandas groupby function.

aggregated_data = results_df.groupby(['stop_type'], sort=True).agg(
    trades=('ticker', 'count'),
    false_negatives=('false_negative', 'sum'),
    avg_max_pct=('max_pct', 'mean'), 
    avg_tail_capture=('tail_pct', 'mean'), 
    avg_position_size=('position_sizing', 'mean'), 
    avg_max_ret=('max_trade_ret', 'mean')
).round(2)

aggregated_data

,trades,false_negatives,avg_max_pct,avg_tail_capture,avg_position_size,avg_max_ret
stop_type,,,,,,
10_pct Stop,33,8,64.07,65.47,19999.48,12813.29
12.5_pct Stop,33,6,72.55,75.49,15999.30,11607.54
15_pct Stop,33,5,76.89,83.31,13333.90,10252.80
20_pct Stop,33,4,83.30,86.40,10000.08,8330.22
25_pct Stop,33,3,89.56,91.50,8000.11,7165.24
30_pct Stop,33,0,95.39,98.37,6666.48,6359.53
8_pct Stop,33,12,51.59,53.08,25000.47,12900.08
Actual Outcome,33,13,30.89,49.46,20305.97,5954.27
Initial Stop,33,9,63.17,64.61,20305.97,13024.68


In [ ]:
# Using pandas functionality to output dataframe to CSV

results_df.to_csv('results.csv', index=False)